<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from config.spark_config import SparkConfig
from config.io_config import *
from app.platform_app import PlatformApp
from transformation.gold_loader import *
from utils.logger import LoggerConfig
from transformation.bronze_ingestion import *
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from utils.data_quality import *
from utils.data_cleaning import *
from utils.utils import *

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Set up</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Set up
    </h1>
</div>


In [2]:
logger = LoggerConfig().setup_logger(log_dir=None)
spark = SparkConfig.create_spark(app_name="FMCG Domain", logger=logger, use_databricks=True)
app = PlatformApp(spark=spark, logger=logger, catalog_name="fmcg_domain")

2026-03-19 19:20:52 | INFO     | Logging configured: level=DEBUG, format=colored
2026-03-19 19:20:53 | INFO     | Connected to Databricks via Spark Connect.
2026-03-19 19:20:53 | INFO     | Initializing Data Platform...
2026-03-19 19:20:53 | INFO     | Spark session initialized


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Bronze</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Bronze
    </h1>
</div>


In [3]:
# Bronze Ingestion
logger.info("Ingesting data into Bronze layer...")
upload_file_to_bronze(spark=spark, logger=logger, entity="customers",
                      path_data=S3_BASE_PATH, path_cp=CP_PATH_BRONZE,
                      name_catalog=app.catalog_name)
logger.info("Bronze ingestion completed.")
logger.info("*" * 80)

2026-03-19 19:20:53 | INFO     | Ingesting data into Bronze layer...
2026-03-19 19:20:53 | INFO     | Starting Bronze ingestion for entity: Customers
2026-03-19 19:20:53 | INFO     | Uploading file: s3://00-sportsbar-dp/customers
2026-03-19 19:21:00 | INFO     | Customers: Schema inferred successfully
2026-03-19 19:21:06 | INFO     | Completed Bronze ingestion for entity: Customers
+-----------+----------------------+---------+----------------------+-------------+---------+----------------------+
|customer_id|customer_name         |city     |read_timestamp        |file_name    |file_size|file_modification_time|
+-----------+----------------------+---------+----------------------+-------------+---------+----------------------+
|789201     |FitFuel Market        |Bengaluru|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789202     |FitFuel Market        |Hyderabad|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789203     |FitFuel Market   

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Silver</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Silver
    </h1>
</div>


In [4]:
df_bronze = spark.sql(f"SELECT * FROM {app.catalog_name}.bronze.customers")
df_bronze.show(truncate=False)

+-----------+----------------------+----------+----------------------+-------------+---------+----------------------+
|customer_id|customer_name         |city      |read_timestamp        |file_name    |file_size|file_modification_time|
+-----------+----------------------+----------+----------------------+-------------+---------+----------------------+
|789201     |FitFuel Market        |Bengaluru |2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789202     |FitFuel Market        |Hyderabad |2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789203     |FitFuel Market        |New Delhi |2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789301     |Athlete's Choice Store|Bengaluru |2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789303     |Athlete's Choice Store|New Delhi |2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789101     |Endurance Foods       |Bengalore |2026-03-1

## Transformations

### Duplicates

In [5]:
df_duplicates = check_duplicate(df=df_bronze, logger=logger)

2026-03-19 19:21:10 | WARNING  | 4 duplicate rows detected (10.26%).
2026-03-19 19:21:10 | WARNING  | Rows affected: 4 out of 39.


In [6]:
df_silver = dedup(df=df_bronze, dedup_cols=["customer_id"], 
                  cdc="read_timestamp", logger=logger)

2026-03-19 19:21:10 | INFO     | Starting deduplication
2026-03-19 19:21:10 | INFO     | Dedup columns: ['customer_id'] | CDC column: read_timestamp
2026-03-19 19:21:10 | INFO     | Input row count: 39
2026-03-19 19:21:11 | INFO     | Output row count after dedup: 35
2026-03-19 19:21:11 | INFO     | Removed duplicate rows: 4
2026-03-19 19:21:11 | INFO     | Deduplication completed


### Check NULL

In [7]:
# check null
check_null(df = df_silver, logger=logger)

2026-03-19 19:21:13 | WARNING  | 
+----------+---------------+-----------+
| Features | Missing_Count | Missing_% |
+----------+---------------+-----------+
|   city   |       4       |   11.43   |
+----------+---------------+-----------+
2026-03-19 19:21:13 | WARNING  | Total missing values: 4 out of 35 rows.


In [8]:
# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    789403: "New Delhi",
    789420: "Bengaluru",
    789521: "Hyderabad",
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

df_fix.show(truncate=False)

+-----------+----------+
|customer_id|fixed_city|
+-----------+----------+
|789403     |New Delhi |
|789420     |Bengaluru |
|789521     |Hyderabad |
|789603     |Hyderabad |
+-----------+----------+



In [9]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn("city", F.coalesce("city", "fixed_city"))  # Replace null with fixed city
    .drop("fixed_city")
)

# Sanity Checks
check_null(df = df_silver, logger=logger)

2026-03-19 19:21:17 | INFO     | No missing values detected in 35 rows.


### Trim spaces in customer name

In [10]:
# check those values
(df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))).show(truncate=False)

+-----------+----------------------+---------+----------------------+-------------+---------+----------------------+
|customer_id|customer_name         |city     |read_timestamp        |file_name    |file_size|file_modification_time|
+-----------+----------------------+---------+----------------------+-------------+---------+----------------------+
|789121     | HydroBoost Nutrition |Hyderabad|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789401     | SprintX nutrition    |Bengaluru|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789420     | ZenAthlete foods     |Bengaluru|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789421     | ZenAthlete Foods     |Hyderbad |2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789521     | PrimeFuel Nutrition  |Hyderabad|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789702     | StaminaX Store       |Hyderabad|2026-03-17 15:41:1

In [11]:
# remove those trim values
df_silver = clean_dataframe(df=df_silver)
df_silver.show(truncate=False)

+-----------+----------------------+----------+----------------------+-------------+---------+----------------------+
|customer_id|customer_name         |city      |read_timestamp        |file_name    |file_size|file_modification_time|
+-----------+----------------------+----------+----------------------+-------------+---------+----------------------+
|789101     |Endurance Foods       |Bengalore |2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789102     |Endurance Foods       |Hyderabad |2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789103     |Endurance Foods       |New Delhi |2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789121     |HydroBoost Nutrition  |Hyderabad |2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789122     |HydroBoost Nutrition  |New Delhi |2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789201     |FitFuel Market        |Bengaluru |2026-03-1

### Data Quality Fix: Correcting City Typos

In [12]:
# Sanity check
df_silver.select("city").distinct().show()

+----------+
|      city|
+----------+
| Bengalore|
| Hyderabad|
| New Delhi|
| Bengaluru|
|Hyderabadd|
|  Hyderbad|
| NewDelhee|
|  NewDelhi|
|Bengaluruu|
|  NewDheli|
+----------+



In [13]:
# Dictionary to correct city name typos
# key   : incorrect value in the dataset
# value : standardized city name
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore' : 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad'  : 'Hyderabad',

    'NewDelhi'  : 'New Delhi',
    'NewDheli'  : 'New Delhi',
    'NewDelhee' : 'New Delhi'
}

# List of valid city values
allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    # Step 1: fix city typos using mapping dictionary
    .replace(city_mapping, subset=["city"])
    # Step 2: validate city values
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)                # keep NULL if city is NULL
         .when(F.col("city").isin(allowed), F.col("city"))  # keep value if it is valid
         .otherwise(None)                                   # set to NULL if invalid
    )
)

df_silver.show(truncate=False)

+-----------+----------------------+---------+----------------------+-------------+---------+----------------------+
|customer_id|customer_name         |city     |read_timestamp        |file_name    |file_size|file_modification_time|
+-----------+----------------------+---------+----------------------+-------------+---------+----------------------+
|789101     |Endurance Foods       |Bengaluru|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789102     |Endurance Foods       |Hyderabad|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789103     |Endurance Foods       |New Delhi|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789121     |HydroBoost Nutrition  |Hyderabad|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789122     |HydroBoost Nutrition  |New Delhi|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789201     |FitFuel Market        |Bengaluru|2026-03-17 15:41:1

### Fix Title-Casing Issue

In [14]:
# sanity check
df_silver.select("customer_name").distinct().show(truncate=False)

+----------------------+
|customer_name         |
+----------------------+
|Endurance Foods       |
|HydroBoost Nutrition  |
|FitFuel Market        |
|MacroBite Superfoods  |
|MacroBite superfoods  |
|Athlete's Choice Store|
|PowerSnack Hub        |
|PowerSnack hub        |
|SprintX nutrition     |
|SprintX Nutrition     |
|ZenAthlete foods      |
|ZenAthlete Foods      |
|Peak performance Store|
|Peak Performance store|
|PrimeFuel Nutrition   |
|Recovery Lane         |
|EliteAthlete Nutrition|
|StaminaX Store        |
|GamePlan Foods        |
|Champion's choice     |
+----------------------+
only showing top 20 rows


In [15]:
#

+----------------------+
|customer_name         |
+----------------------+
|Endurance Foods       |
|Hydroboost Nutrition  |
|Fitfuel Market        |
|Macrobite Superfoods  |
|Athlete's Choice Store|
|Powersnack Hub        |
|Sprintx Nutrition     |
|Zenathlete Foods      |
|Peak Performance Store|
|Primefuel Nutrition   |
|Recovery Lane         |
|Eliteathlete Nutrition|
|Staminax Store        |
|Gameplan Foods        |
|Champion's Choice     |
+----------------------+



### Convert customer_id to string

In [16]:
df_silver.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)
 |-- file_modification_time: timestamp (nullable = true)



In [17]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
df_silver.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)
 |-- file_modification_time: timestamp (nullable = true)



### Standardizing Customer Attributes to Match Parent Company Data Model

In [18]:
df_silver.show(truncate=False)

+-----------+----------------------+---------+----------------------+-------------+---------+----------------------+
|customer_id|customer_name         |city     |read_timestamp        |file_name    |file_size|file_modification_time|
+-----------+----------------------+---------+----------------------+-------------+---------+----------------------+
|789101     |Endurance Foods       |Bengaluru|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789102     |Endurance Foods       |Hyderabad|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789103     |Endurance Foods       |New Delhi|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789121     |Hydroboost Nutrition  |Hyderabad|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789122     |Hydroboost Nutrition  |New Delhi|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |
|789201     |Fitfuel Market        |Bengaluru|2026-03-17 15:41:1

In [19]:
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

df_silver.show(truncate=False)

+-----------+----------------------+---------+----------------------+-------------+---------+----------------------+--------------------------------+------+----------+-----------+
|customer_id|customer_name         |city     |read_timestamp        |file_name    |file_size|file_modification_time|customer                        |market|platform  |channel    |
+-----------+----------------------+---------+----------------------+-------------+---------+----------------------+--------------------------------+------+----------+-----------+
|789101     |Endurance Foods       |Bengaluru|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |Endurance Foods-Bengaluru       |India |Sports Bar|Acquisition|
|789102     |Endurance Foods       |Hyderabad|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |Endurance Foods-Hyderabad       |India |Sports Bar|Acquisition|
|789103     |Endurance Foods       |New Delhi|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03

### Transformed data to Silver Layer

In [20]:
if not spark.catalog.tableExists(SILVER_CUSTOMER):
    logger.info("Silver customers table not found. Creating new table...")
    df_silver.write.format("delta") \
                   .option("delta.enableChangeDataFeed", "true") \
                   .option("mergeSchema", "true") \
                   .mode("overwrite") \
                   .saveAsTable(SILVER_CUSTOMER)
    logger.info("Silver customers table created successfully")
else:
    logger.info("Silver customers table exists. Performing upsert...")
    upsert(spark=spark, df=df_silver, key_cols=["customer_id"],
           table="dim_customers", cdc="read_timestamp",
           name_catalog=app.catalog_name, name_schema="silver", logger=logger)
    logger.info("Upsert completed successfully")

2026-03-19 19:21:25 | INFO     | Silver customers table exists. Performing upsert...
2026-03-19 19:21:25 | INFO     | Starting UPSERT into fmcg_domain.silver.dim_customers
2026-03-19 19:21:32 | INFO     | UPSERT completed successfully: fmcg_domain.silver.dim_customers
2026-03-19 19:21:32 | INFO     | Upsert completed successfully


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Gold</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Gold
    </h1>
</div>


In [21]:
df_silver = spark.sql(f"SELECT * FROM {app.catalog_name}.silver.dim_customers")
df_silver.show(truncate=False)

+-----------+----------------------+---------+----------------------+-------------+---------+----------------------+--------------------------------+------+----------+-----------+
|customer_id|customer_name         |city     |read_timestamp        |file_name    |file_size|file_modification_time|customer                        |market|platform  |channel    |
+-----------+----------------------+---------+----------------------+-------------+---------+----------------------+--------------------------------+------+----------+-----------+
|789903     |Champion's Choice     |New Delhi|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |Champion's Choice-New Delhi     |India |Sports Bar|Acquisition|
|789902     |Champion's Choice     |Hyderabad|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03-05 13:07:42   |Champion's Choice-Hyderabad     |India |Sports Bar|Acquisition|
|789721     |Gameplan Foods        |Hyderabad|2026-03-17 15:41:19.78|customers.csv|1404     |2026-03

In [22]:
# take req cols only
# "customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")
df_gold = process_timestamp(df=df_gold)
df_gold.show(truncate=False)

+-----------+----------------------+---------+--------------------------------+------+----------+-----------+--------------------------+
|customer_id|customer_name         |city     |customer                        |market|platform  |channel    |process_timestamp         |
+-----------+----------------------+---------+--------------------------------+------+----------+-----------+--------------------------+
|789903     |Champion's Choice     |New Delhi|Champion's Choice-New Delhi     |India |Sports Bar|Acquisition|2026-03-19 12:21:28.806895|
|789902     |Champion's Choice     |Hyderabad|Champion's Choice-Hyderabad     |India |Sports Bar|Acquisition|2026-03-19 12:21:28.806895|
|789721     |Gameplan Foods        |Hyderabad|Gameplan Foods-Hyderabad        |India |Sports Bar|Acquisition|2026-03-19 12:21:28.806895|
|789720     |Gameplan Foods        |Bengaluru|Gameplan Foods-Bengaluru        |India |Sports Bar|Acquisition|2026-03-19 12:21:28.806895|
|789703     |Staminax Store        |New D

In [23]:
logger.info("Gold customers table not found. Creating new table...")
df_gold.write.format("delta") \
                .option("delta.enableChangeDataFeed", "true") \
                .option("mergeSchema", "true") \
                .mode("overwrite") \
                .saveAsTable(GOLD_SB_CUSTOMER)
logger.info("Gold customers table created successfully")

2026-03-19 19:21:34 | INFO     | Gold customers table not found. Creating new table...
2026-03-19 19:21:36 | INFO     | Gold customers table created successfully


## Merging Data source with parent

In [24]:
df_gold = spark.sql(f"SELECT * FROM {app.catalog_name}.gold.dim_customers")
df_gold.count()

18

In [29]:
delta_table = DeltaTable.forName(spark, GOLD_CUSTOMER)
df_child_customers = spark.table(GOLD_SB_CUSTOMER).select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

,num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,35,0,0,35


In [30]:
df_gold = spark.sql(f"SELECT * FROM {app.catalog_name}.gold.dim_customers")
df_gold.count()

53

In [31]:
app.stop()

2026-03-19 19:27:37 | INFO     | Stopping Spark session...
2026-03-19 19:27:38 | INFO     | Spark stopped.
